# ALiBi (Attention with Linear Biases) from scratch

Walkthrough for **ai-learn-27-alibi-attention**.

ALiBi adds a static linear penalty `-m_h · |i−j|` to attention logits so heads prefer nearer keys — no learned positional embeddings, and the bias extends to any length (Press et al. 2021).


## 1. Geometric head slopes

For a power-of-two head count `H`, slopes are a geometric sequence starting at `2^{-8/H}`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from alibi import get_alibi_slopes, check_slope_sanity, alibi_bias

print("H=4", get_alibi_slopes(4))
print("H=8", get_alibi_slopes(8))
print(check_slope_sanity(8))


## 2. Bias matrix

`bias[i,j] = -m_h · |i−j|` (bidirectional). Causal mode uses `(i−j)` below the diagonal.


In [ ]:
import numpy as np
slopes = get_alibi_slopes(4)
bias = alibi_bias(8, slopes, causal=False)
print("slopes", slopes)
print("head0 bias:\n", np.round(bias[0], 3))


## 3. Locality of attention mass

With random Q/K, ALiBi puts more mass on nearby keys than no-PE.


In [ ]:
from experiments import locality_attention_mass

loc = locality_attention_mass(seed=42)
for pe in ("none", "absolute", "alibi"):
    print(f"{pe:10s} near_mass={loc[pe]['mean_near_mass']} entropy={loc[pe]['mean_entropy']}")


## 4. Length extrapolation

Train near-neighbor retrieval at `L_train=16`, test at longer lengths. ALiBi holds; absolute PE drops.


In [ ]:
from experiments import length_extrapolation

for pe in ("none", "absolute", "alibi"):
    r = length_extrapolation(pe, seed=42)
    lens = ", ".join(f"L{b['seq_len']}={b['acc']}" for b in r["by_length"])
    print(f"{pe:10s} train={r['train_acc']}  {lens}")


## 5. Full smoke

```bash
python run_smoke.py   # writes results/ (seed 42)
```

See `results/RESULTS.md` for tables and SVG plots.

### ALiBi vs RoPE (conceptually)
| | ALiBi | RoPE |
|---|---|---|
| Where | add bias to logits | rotate Q/K pairs |
| Form | linear distance penalty | phase rotation by `m·θ` |
| Params | none (fixed slopes) | none (fixed frequencies) |
| Strength | simple length extrapolation | rich relative inner-product geometry |
